In [1]:
import pandas as pd

df = pd.read_parquet('fhv_tripdata_2023-01.parquet')

print(df.shape)
print(df.dtypes)

(1114320, 7)
dispatching_base_num                 str
pickup_datetime           datetime64[us]
dropOff_datetime          datetime64[us]
PUlocationID                     float64
DOlocationID                     float64
SR_Flag                           object
Affiliated_base_number               str
dtype: object


In [2]:
print(df.head(10))
print(df.isnull().sum())

  dispatching_base_num     pickup_datetime    dropOff_datetime  PUlocationID  \
0               B00008 2023-01-01 00:30:00 2023-01-01 01:00:00           NaN   
1               B00078 2023-01-01 00:01:00 2023-01-01 03:15:00           NaN   
2               B00111 2023-01-01 00:30:00 2023-01-01 01:05:00           NaN   
3               B00112 2023-01-01 00:34:45 2023-01-01 00:52:03           NaN   
4               B00112 2023-01-01 00:11:20 2023-01-01 00:22:03           NaN   
5               B00112 2023-01-01 00:33:28 2023-01-01 00:53:46           NaN   
6               B00112 2023-01-01 00:33:11 2023-01-01 00:48:45           NaN   
7               B00112 2023-01-01 00:55:24 2023-01-01 01:02:55           NaN   
8               B00112 2023-01-01 00:39:16 2023-01-01 00:39:23           NaN   
9               B00112 2023-01-01 00:50:10 2023-01-01 00:50:17           NaN   

   DOlocationID SR_Flag Affiliated_base_number  
0           NaN    None                 B00008  
1           NaN    No

In [3]:
df['duracion_min'] = (df['dropOff_datetime'] - df['pickup_datetime']).dt.total_seconds() / 60

print(df['duracion_min'].describe())

count    1.114320e+06
mean     2.515179e+01
std      5.910555e+02
min      1.666667e-02
25%      9.616667e+00
50%      1.700000e+01
75%      2.958333e+01
max      4.377832e+05
Name: duracion_min, dtype: float64


In [4]:
viajes_sucios = df[(df['duracion_min'] < 1) | (df['duracion_min'] > 300)]
print(f"Viajes sospechosos: {len(viajes_sucios)}")
print(f"Porcentaje del total: {round(len(viajes_sucios) / len(df) * 100, 2)}%")

Viajes sospechosos: 11729
Porcentaje del total: 1.05%


In [5]:
df_limpio = df[(df['duracion_min'] >= 1) & (df['duracion_min'] <= 300)]
print(f"Filas originales: {len(df)}")
print(f"Filas limpias {len(df_limpio)}")

Filas originales: 1114320
Filas limpias 1102591


# Preguntas de negocio
# 1. ¿Cuáles son las horas del día con más viajes?
# 2. ¿Cuáles zonas de destino son las más populares?
# 3. ¿Cuál empresa despacha más viajes?
# 4. ¿En qué días de la semana son los viajes más largos?

In [6]:
df_limpio['hora'] = df_limpio['pickup_datetime'].dt.hour

viajes_por_hora = df_limpio.groupby('hora').size().reset_index(name='total_viajes')
print(viajes_por_hora)

    hora  total_viajes
0      0         16281
1      1         10841
2      2          8080
3      3          8984
4      4         16243
5      5         23925
6      6         37870
7      7         63805
8      8         80712
9      9         87925
10    10         85407
11    11         78717
12    12         78778
13    13         76171
14    14         71905
15    15         63348
16    16         56378
17    17         48597
18    18         40706
19    19         35750
20    20         32913
21    21         29846
22    22         26869
23    23         22540


In [9]:
df_limpio['dia_semana'] = df_limpio['pickup_datetime'].dt.day_name()

duracion_por_dia = df_limpio.groupby('dia_semana')['duracion_min'].mean().round(2).reset_index()
print(duracion_por_dia)

  dia_semana  duracion_min
0     Friday         24.21
1     Monday         22.77
2   Saturday         19.74
3     Sunday         19.32
4   Thursday         25.54
5    Tuesday         24.71
6  Wednesday         24.72
